### 수동 보정 시 발견된 예외 케이스 및 처리 로직

수동 보정 과정에서 웹 UI의 표준 필드에 데이터가 누락되거나 왜곡된 경우 아래의 예외 처리 규칙에 따라 데이터를 보정하였습니다. 본 가이드는 향후 데이터 추출 로직 설계 및 데이터 클리닝의 최우선 지침으로 활용됩니다.

| 예외 유형 | 발생 현상 및 원인 | 보정 규칙 (Logic) |
| --- | --- | --- |
| **공개일자 데이터 불일치** | 공개일자와 실제 입찰 참여 시작일의 격차가 크거나 시스템 필드 값이 부정확함 | **처리**: 공개일자는 개별 수집 항목에서 **제외**함. 대신 데이터 수집 시 **검색 범위(1년 단위)** 를 설정하기 위한 필터링 조건으로만 활용함 |
| **데이터 신뢰성 불일치** | 발주기관 메타데이터와 파일 본문 내용이 상이한 경우 | **처리**: 파일 본문과 발주기관을 교차 검증하여 일치하지 않는 파일은 제거(Drop)함 |
| **사업비 특이 케이스 (0/1)** | 시스템상 예산이 누락되었거나 비공개인 경우 | **0 (미기입)**: 상세 페이지 내 **용역비용** 등에서 찾아 실금액으로 수정 <br> **1 (비공개)**: 공고문 내에서도 확인 불가능한 비공개 건은 **1**로 기입하여 관리 |
| **사업비 우선순위** | 배정예산과 추정가격이 동시에 존재하는 경우 | **배정예산(VAT 포함)** 을 최우선으로 기록하여 실제 총 사업 규모 반영 |
| **추정가격만 존재** | 배정예산 필드가 비어 있고 추정가격만 명시된 경우 | **추정가격**을 기록하되 비고란에 **VAT 제외 금액**임을 명시 |
| **입찰한도액 표기** | 사업 금액 항목 대신 입찰한도액으로 명시된 경우 | **용역규모**: 입찰한도액을 해당 사업의 예산(숫자)으로 기록 |
| **정정공고** | 내용 오류 수정이나 일정 변경으로 수정 공고가 올라온 경우 | **번호**: 최신 정정 번호 적용 <br>**시작일**: **최초 공고 게시일** 적용 <br>**종료일**: **정정된 최종 마감일** 적용 |
| **재공고 사업** | 유찰 등으로 동일 사업이 다시 공고된 경우 | **번호**: 최신 재공고 번호 적용 <br>**시작일**: **최초 공고 게시일** 적용 <br>**종료일**: **최종 재공고의 마감일** 적용 |
| **직찰/집찰 공고** | 입찰 방법 특성상 시스템 날짜 필드가 누락된 경우 | **시작일**: 상세 일정 테이블의 **공고 게시** 일시 적용<br>**종료일**: 상세 일정 테이블의 **개찰** 또는 **마감** 일시 적용 |
| **공고번호 왜곡** | 11자리 이상의 번호가 엑셀에서 지수(E+) 형태로 변형됨 | **처리**: 셀 서식을 **텍스트(Text)** 로 강제 지정하여 데이터 유실 및 뒷자리 변조 방지 |
| **공고서 참조 일정** | 시스템 필드 대신 첨부파일 내부에만 일정이 명시된 경우 | **공고서(PDF/HWP)** 본문 내 일정 테이블을 참조하여 **제안서 마감일**을 종료일로 설정 |
| **검색어 불일치** | 사업명만으로 검색 시 유사 공고가 너무 많이 노출되는 경우 | **공고기관명 + 사업명** 조합으로 검색어를 재구성하여 정확도 향상 |

**비고 및 인사이트**

- **날짜 기준 정립**: 공개일자는 검색 범위를 1년 단위로 한정하는 용도로만 사용하며 실제 분석용 시작일은 상세 일정의 **공고 게시** 일자를 기준으로 함
- **신뢰성 검증**: 발주기관과 파일 본문이 다른 문서는 신뢰도 문제로 데이터셋에서 즉시 제외함
- **사업비 구분**: **0**은 보정 대상 데이터 **1**은 분석 제외 또는 비공개 데이터로 분류하여 통계의 정확성을 확보함

**데이터 무결성 주의 사항**

- **공고번호 보존**: 11자리 이상의 번호는 지수 형태로 저장 시 뒷자리가 유실되므로 반드시 엑셀 셀 왼쪽 상단의 **초록색 삼각형**이 보이는 **텍스트 상태**로 관리함
- **파이프라인 연동**: CSV 로드 시 공고번호 컬럼을 **문자열(str)** 타입으로 지정하여 데이터 왜곡을 방지함
- **통계 예외 처리**: 사업비가 **0** 또는 **1**로 기록된 데이터는 평균이나 합계 계산 시 별도의 필터링 처리가 필요함

### 4. 데이터 제외 리스트 (Data Exclusion List)

- 데이터 신뢰성 확보를 위해 분석 및 학습 데이터셋에서 제외된 항목들입니다. 외부 공유를 위해 주요 정보는 비식별화 처리되었습니다.

| 발주 기관 | 사업명 | 파일명 | 제외 사유 |
| --- | --- | --- | --- |
| **********스 | 통합정보시스템 고도화 용역 | **********스_통합정보시스템 고도화 용역.hwp | 타 연구기관(*******원)의 발주 파일과 본문 내용이 완전히 일치하며, 조달 시스템상에서도 타 기관의 실적으로만 검색되어 데이터 신뢰성 결여로 판단됨 |

**데이터 제외의 목적 및 RAG 시스템 영향**

- **데이터 오염 방지**: 발주 기관 메타데이터와 실제 파일 내용이 상이할 경우, LLM이 잘못된 정보원을 근거로 답변을 생성하는 **환각(Hallucination)** 현상을 초래함.
- **그라운딩(Grounding) 강화**: RAG 시스템의 핵심은 정확한 출처 기반의 답변 생성임. 신뢰할 수 없는 데이터는 검색 결과의 품질을 저하시키므로 원천 차단이 필요함.
- **필터링 정합성**: 기관별 예산이나 사업 규모를 분석할 때 잘못된 메타데이터가 포함되면 전체 통계 데이터의 **무결성**이 파괴됨.

**비식별화 및 보안 지침**

- **기관명 마스킹**: 특정 기관을 유추할 수 있는 고유 명칭은  처리함.
- **본문 식별**: 파일 내부의 도장, 인장, 워터마크 등을 통해 실제 소유 기관을 판별한 후, 메타데이터와 불일치할 경우 즉시 제외 리스트에 등록함.
- **내부 관리**: 실제 삭제 작업을 수행하는 엔지니어는 팀 내부 보안 드라이브의 **원본 대조표**를 참조하여 작업을 수행함.